In [ ]:
import os
from copy import deepcopy
import numpy as np
import random
import torch
random_seed = 2025
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Replace "0" with the desired GPU device index
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from utils import calc_logit_norm

In [ ]:
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),  # converts to tensor and scales image pixel values to [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # normalize using MNIST's mean and std
])

train_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=True, download=False, transform=transform)
test_dataset  = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=transform)


In [ ]:
# Define the CNN model with two Batch Normalization layers
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # First convolutional block
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional block
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.bn2   = nn.BatchNorm2d(64)
        
        # Fully connected layers
        # After two conv layers and pooling operations, the spatial dimensions reduce.
        # MNIST images are 28x28. After two rounds of 3x3 conv (without padding) and 2x2 pooling:
        #   After conv1: 28-3+1 = 26 -> after pool: 26/2 = 13 (floor division)
        #   After conv2: 13-3+1 = 11 -> after pool: 11/2 = 5 (floor division)
        # Thus, the feature map size is 64 x 5 x 5.
        self.fc1 = nn.Linear(64 * 5 * 5, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        # First conv block: Conv -> BatchNorm -> ReLU -> Pool
        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.pool(x)
        
        # Second conv block: Conv -> BatchNorm -> ReLU -> Pool
        x = self.conv2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.pool(x)
        
        # Flatten the tensor for the fully connected layers
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
# test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# # Initialize the model, define loss function and optimizer
# model = CNN().to('cuda')
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# # Training loop
# num_epochs = 5
# model.train()
# for epoch in range(num_epochs):
#     running_loss = 0.0
#     for images, labels in train_loader:
#         # Zero the parameter gradients
#         optimizer.zero_grad()
        
#         # Forward pass
#         outputs = model(images.to('cuda'))
#         loss = criterion(outputs, labels.to('cuda'))
        
#         # Backward pass and optimization
#         loss.backward()
#         optimizer.step()
        
#         running_loss += loss.item()
#     print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
# torch.save(model, '/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model.pth')

In [ ]:
model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model.pth')

In [ ]:
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

def validate(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to('cuda'), labels.to('cuda')
            outputs = model(images)
            # Get predictions from the maximum value
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy

print(f"Test Accuracy: {validate(model, test_loader): .2%}")

In [ ]:

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap

def _visualize_features(features, labels, title="t-SNE Visualization of CNN Feature Outputs"):
    # Apply t-SNE to reduce dimensionality to 2D
    tsne = TSNE(n_components=2, random_state=42)

    features_tsne = tsne.fit_transform(features)

    # Plot the t-SNE results
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(features_tsne[:, 0], features_tsne[:, 1], c=labels, cmap='tab10', alpha=0.7)
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.title(title)
    plt.colorbar(scatter, ticks=range(10), label='Digit Label')
    plt.show()

def visualize_features(features, labels, title="Umap Visualization of CNN Feature Outputs", reducer=None):
    
    if reducer is None:
        reducer = umap.UMAP(random_state=42)
        embedding = reducer.fit_transform(features)
    else:
        embedding = reducer.transform(features)

    # Plot the UMAP results
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap='tab10', alpha=0.7)
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.title(title)
    plt.colorbar(scatter, ticks=range(10), label='Digit Label')
    plt.show()
    return reducer

def extract_features(model, data_loader):
    # Create a list to store the features and labels
    fc_features = []
    fc_labels = []

    # Define a forward hook to capture the output of fc1 (the hidden fully connected layer)
    def hook(module, input, output):
        # Append the output features and corresponding labels
        fc_features.append(output.detach().cpu())

    # Register the hook on the first fully connected layer (fc1)
    hook_handle = model.fc1.register_forward_hook(hook)

    # Make sure the model is in evaluation mode
    model.eval()

    # Iterate over the test loader to extract features
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to('cuda')
            labels = labels.to('cuda')
            # Forward pass (the hook will capture fc1 activations)
            outputs = model(images)
            # Save the labels for each batch
            fc_labels.extend(labels.cpu().numpy())

    hook_handle.remove()

    # Concatenate all feature batches
    features_array = torch.cat(fc_features, dim=0).numpy()
    return features_array, fc_labels


features, feature_labels = extract_features(model, test_loader)
reducer = visualize_features(features, feature_labels, "Clean Features")


In [ ]:
def add_gaussian_noise(X, severity=5):
    scale = [.08, .12, 0.18, 0.26, 0.38][severity - 1]
    # Add Gaussian noise to the data
    generator = torch.Generator().manual_seed(random_seed)
    noise = torch.normal(size=X.shape, std=scale, mean=0.0, generator=generator)
    noisy_X = X + noise
    return torch.clamp(noisy_X, 0., 1.)

noisy_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: add_gaussian_noise(x, severity=5)),
    transforms.Normalize((0.1307,), (0.3081,))
])

noisy_test_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=noisy_transform)
noisy_test_loader = DataLoader(noisy_test_dataset, batch_size=64, shuffle=False)

In [ ]:
noisy_features, noisy_feature_labels = extract_features(model, noisy_test_loader)
# _ = visualize_features(noisy_features, noisy_feature_labels, "Noisy Features", reducer=reducer)

In [ ]:
# validate noisy data
zero_shot_accuracy = validate(model, noisy_test_loader)
print(f"Test Accuracy: {zero_shot_accuracy: .2%}")

In [ ]:
# visualize clean images and noisy images
def visualize_images(images, labels, n_images=10):
    fig, axes = plt.subplots(n_images, 2, figsize=(10, 20))
    for i in range(n_images):
        axes[i, 0].imshow(images[i].squeeze(), cmap='gray')
        axes[i, 0].axis('off')
        axes[i, 0].set_title(f"Label: {labels[i]} (Clean)")
        
        axes[i, 1].imshow(noisy_test_dataset[i][0].squeeze(), cmap='gray')
        axes[i, 1].axis('off')
        axes[i, 1].set_title(f"Label: {labels[i]} (Noisy)")
    plt.tight_layout()
    plt.show()

visualize_images(test_dataset.data, test_dataset.targets, n_images=10)

In [ ]:
# 5. TENT adaptation: update only BatchNorm layers using entropy minimization
def entropy_loss(logits):
    return -(logits.softmax(1) * logits.log_softmax(1)).sum(1)


def collect_params(model, freeze_layers=[]):
    """Collect the affine scale + shift parameters from batch norms.
    Walk the model's modules and collect all batch normalization parameters.
    Return the parameters and their names.
    Note: other choices of parameterization are possible!
    """
    params = []
    names = []
    for nm, m in model.named_modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.LayerNorm, nn.GroupNorm)):
            if any([f'{layer}' in nm for layer in freeze_layers]):
                continue
            for np, p in m.named_parameters():
                if np in ['weight', 'bias']:  # weight is scale, bias is shift
                    params.append(p)
                    names.append(f"{nm}.{np}")
    return params, names

def configure_model(model):
    """Configure model for use with eata."""
    # train mode, because eata optimizes the model to minimize entropy
    # self.model.train()
    model.eval()  # eval mode to avoid stochastic depth in swin. test-time normalization is still applied
    # disable grad, to (re-)enable only what eata updates
    model.requires_grad_(False)
    # configure norm for eata updates: enable grad + force batch statisics
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.requires_grad_(True)
            # force use of batch stats in train and eval modes
            m.track_running_stats = False
            m.running_mean = None
            m.running_var = None
        elif isinstance(m, nn.BatchNorm1d):
            m.train()   # always forcing train mode in bn1d will cause problems for single sample tta
            m.requires_grad_(True)
            
        elif isinstance(m, (nn.LayerNorm, nn.GroupNorm)):
            m.requires_grad_(True)



In [ ]:
model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model.pth')
configure_model(model)
bn_params, bn_names = collect_params(model, freeze_layers=["bn100"])
print(bn_names)
tent_optimizer = optim.Adam(bn_params, lr=1e-3)

adaptation_loader = noisy_test_loader

test_acc_list = []
logit_norm_list = []

test_acc_list.append((zero_shot_accuracy,0))


epochs = 10
steps = 0

total_steps = len(adaptation_loader) * epochs
log_frequency = 100

for epoch in range(epochs):

    for x_data, y_data in adaptation_loader:
        tent_optimizer.zero_grad()
        x_data, y_data = x_data.to('cuda'), y_data.to('cuda')
        logits = model(x_data)
        loss = entropy_loss(logits).mean(0)
        loss.backward()
        tent_optimizer.step()


        if (steps+1) % log_frequency == 0:
            # test accuracy
            test_accuracy = validate(deepcopy(model), noisy_test_loader)
            test_acc_list.append((test_accuracy, steps))
            print(f"Adaptation step {steps}/{total_steps}, Entropy Loss: {loss.item():.4f}")
            print(f"Test Accuracy: {test_accuracy:.2f}")

            
            logit_norm, _ = calc_logit_norm(torch.Tensor(logits[:,[0]]))
            logit_norm_list.append((logit_norm, steps))

        steps += 1

# plot test accuracy
plt.figure(figsize=(6,3))
step_list = [x[1] for x in test_acc_list]
accuracy_list = [x[0] for x in test_acc_list]
plt.plot(step_list, accuracy_list)
plt.xlabel("Adaptation Steps")
plt.ylabel("Test Accuracy")

plt.figure(figsize=(6,3))
logit_norms = [x[0] for x in logit_norm_list]
plt.plot(step_list[1:], logit_norms)
plt.xlabel("Adaptation Steps")
plt.ylabel("Logit Norm")

In [ ]:
featues_after_adaptation, noisy_feature_labels = extract_features(deepcopy(model), noisy_test_loader)
# _ = visualize_features(featues_after_adaptation, noisy_feature_labels, "Noisy Features after TENT Adaptation", reducer=reducer)

## Imbalanced Adaptation

### Imbalanced Dataloader

In [ ]:
from utils import generate_oversample_indices, class_counts
from torch.utils.data import Subset
from torch.utils.data.sampler import WeightedRandomSampler

mnist_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=transform)

partial_classes = [0]
# Get indices of the dataset that correspond to the desired classes
subset_indices = [i for i, (_, label) in enumerate(mnist_dataset) if label in partial_classes]

# Create a subset dataset using the filtered indices
subset_dataset = Subset(mnist_dataset, subset_indices)

# Calculate class frequencies in the subset dataset
labels = [mnist_dataset[i][1] for i in subset_indices]

sample_weights = [1.0 for label in labels]

# Create a WeightedRandomSampler
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=10000, replacement=True)

# Create DataLoader with the sampler for oversampling
imbalance_dataloader = DataLoader(subset_dataset, batch_size=64, sampler=sampler)

class_counts(imbalance_dataloader)

In [ ]:
model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model.pth')
configure_model(model)
bn_params, bn_names = collect_params(model, freeze_layers=["bn100"])
tent_optimizer = optim.Adam(bn_params, lr=1e-3)


adaptation_loader = imbalance_dataloader

test_acc_list = []
logit_norm_list = []

test_acc_list.append((zero_shot_accuracy,0))


epochs = 10
steps = 0

total_steps = len(adaptation_loader) * epochs
log_frequency = 100

for epoch in range(epochs):

    for x_data, y_data in adaptation_loader:
        tent_optimizer.zero_grad()
        x_data, y_data = x_data.to('cuda'), y_data.to('cuda')
        logits = model(x_data)
        loss = entropy_loss(logits).mean(0)
        loss.backward()
        tent_optimizer.step()


        if (steps+1) % log_frequency == 0:
            # test accuracy
            test_accuracy = validate(deepcopy(model), noisy_test_loader)
            test_acc_list.append((test_accuracy, steps))
            print(f"Adaptation step {steps}/{total_steps}, Entropy Loss: {loss.item():.4f}")
            print(f"Test Accuracy: {test_accuracy:.2f}")

            
            logit_norm, _ = calc_logit_norm(torch.Tensor(logits[:,[0]]))
            logit_norm_list.append((logit_norm, steps))

        steps += 1

# plot test accuracy
plt.figure(figsize=(6,3))
step_list = [x[1] for x in test_acc_list]
accuracy_list = [x[0] for x in test_acc_list]
plt.plot(step_list, accuracy_list)
plt.xlabel("Adaptation Steps")
plt.ylabel("Test Accuracy")

plt.figure(figsize=(6,3))
logit_norms = [x[0] for x in logit_norm_list]
plt.plot(step_list[1:], logit_norms)
plt.xlabel("Adaptation Steps")
plt.ylabel("Logit Norm")

In [ ]:
featues_after_adaptation, noisy_feature_labels = extract_features(deepcopy(model), noisy_test_loader)
_ = visualize_features(featues_after_adaptation, noisy_feature_labels, "Noisy Features after Imbalance Adaptation", reducer=reducer)

In [ ]:
import matplotlib.animation as animation

# Simulated data: 50 frames, 100 points per frame
num_frames = 50
num_points = 100
np.random.seed(42)
initial_embedding = np.random.rand(num_points, 2)

# Simulate class labels for each point (e.g., 3 classes)
labels = np.random.randint(0, 3, size=num_points)

# Create a list of embeddings showing drift over time.
umap_embeddings = [
    initial_embedding + 0.05 * frame * np.random.randn(num_points, 2)
    for frame in range(num_frames)
]

# Set up the figure and initial scatter plot with colors defined by labels.
fig, ax = plt.subplots()
scat = ax.scatter(
    umap_embeddings[0][:, 0],
    umap_embeddings[0][:, 1],
    c=labels,          # Points colored by their class labels.
    cmap='tab10',    # You can choose any colormap.
    s=20
)
ax.set_xlim(0, 1.5)
ax.set_ylim(0, 1.5)
ax.set_title("UMAP Feature Drift with Class Colors")

# Update function for animation: update point positions per frame.
def update(frame):
    data = umap_embeddings[frame]
    scat.set_offsets(data)
    ax.set_title(f"Frame {frame+1}")
    return scat,

# Create the animation using FuncAnimation.
ani = animation.FuncAnimation(fig, update, frames=num_frames, interval=1000, blit=True)
# Save the animation as a video file (requires ffmpeg installed)
# Writer = animation.FFMpegWriter
# writer = Writer(fps=5, metadata=dict(artist='Your Name'), bitrate=1800)
ani.save("umap_feature_drift.gif")

plt.show()